# ReFuelEU optimisation — runs

Every AeroMAPS run behind the paper, then the dataset `02_results.ipynb` reads. Each cell
calls one driver in `runs/`. The problem itself (scenario, design variables, constraints
G1–G6) is built in `helpers/optimisation_runs.py` from the configurations in `data/`; the
drivers set SLSQP to stop on the KKT residual (relative tolerance 1e-6, at most 50
iterations).

All drivers are **resumable**: a run already on disk is skipped, so a cell can be
re-executed after an interruption, and deleting a run's files makes it run again. Raw
outputs (one JSON of model outputs, plus one HDF of the optimisation history) go to
`runs/results_sensitivities/` and `runs/results_carbon_biomass_grid/`, and are not
committed (~50 MB). Each optimisation logs to `runs/logs/<run>.log`.

The cells can equally be run from a terminal, from this folder, by dropping `!{PYTHON}`
for `python`. That is the better choice for the long ones, which then survive the
notebook (e.g. `nohup python runs/run_block_e.py ... &`).

**The five cases**, differing only in two hypotheses:

| case | biomass to aviation | aircraft efficiency gain |
|---|---|---|
| `main` | 10 % of world biomass | 1.35 %/yr |
| `B5`, `B75`, `B15` | 5, 7.5, 15 % | 1.35 %/yr |
| `pess` | 10 % | 0.91 %/yr (no new aircraft) |

In [ ]:
import sys

PYTHON = sys.executable

## 1. ReFuelEU references and the carbon budget

Three plain MDAs at 10 % biomass, under a minute: ReFuelEU interpolated linearly between
its milestones, ReFuelEU held in steps as legislated, and the fossil BAU. The cumulative
2020–2050 CO₂ of the linear one, **3.8656 GtCO₂** (3.1226 % of the world budget), is the
budget blocks A–D and section 4 are held to; the script checks it against the value
hard-coded in `run_batch.py`. It also checks whether ReFuelEU itself satisfies G2–G6 (it
breaches the biofuel ramp-up in 2035, §4.1), under the baseline and the block B ramp-up
caps.

Writes `runs/preflight_budget.csv` and `runs/preflight_refueleu_feasibility.csv`.

In [ ]:
!{PYTHON} runs/preflight.py

## 2. Blocks A–D: one parameter at a time

Fourteen optimisations at the ReFuelEU-equivalent budget: `base`, then price elasticity
(A, plus fixed demand A′), ramp-up rate (B) and volume (B′) caps, dedicated-wind
electrofuel (C) and discount rate (D). Between 1 and 26 minutes each, about 40 minutes at
three in parallel.

Each run starts from its published start point in `runs/previous_optima.csv`, so all
fourteen are independent. Solver diagnostics (stop criterion, evaluations, largest
constraint violation, time) are collected in `runs/summary.csv`.

Options:
- `--jobs N` runs N at a time;
- `--list` shows the matrix and what is already done;
- named runs only, e.g. `run_batch.py --jobs 2 r_7 r_15`;
- `AEROMAPS_SEED=0` ignores the start points: `base` then runs first from a cold start
  and every other run starts from its optimum;
- `python runs/status.py` reports progress from another terminal.

In [ ]:
!{PYTHON} runs/run_batch.py --jobs 3

## 3. Block E: the carbon-budget ladder of each case

For each of the five cases, the optimum at every budget from 3.8 down to 2.0 % of the
world budget, plus the minimum-CO₂ point (the budget becomes the objective), and the fossil
BAU and ReFuelEU references: 55 optimisations, several hours. `B15` is the slowest.

With `--seed runs/block_e_seeds.csv`, as published, every (case, budget) starts from its
published start point and is an independent job. Without it, each case is a continuation
chain, loosest budget first, each run starting from the previous optimum, so budgets run in
sequence and only cases run in parallel.

The tightest budgets are infeasible by construction, the ramp-up limits capping how fast
the fleet can decarbonise: `B5` from 2.4 %, `B75` and `pess` from 2.2 %, `main` and `B15`
at 2.0 %. Their `feasible` flag is `False` in the dataset.

Options: `--jobs N`; case names to restrict the sweep, e.g. `main B15`;
`--status` prints the case × budget grid of what is done, running or infeasible.

In [ ]:
!{PYTHON} runs/run_block_e.py --jobs 4 --seed runs/block_e_seeds.csv

## 4. Each case at the ReFuelEU-equivalent budget

The five cases optimised at the budget of section 2, each warm-started from its own
3.2 % rung (so block E must be done). `opt_main_refueleu` is the optimum discussed in §4.1;
the four others are the biomass and technology rows of Figure 8. For `main`, the script
also compares the result with `base`, the same problem reached from another start point:
they agree to 1e-4 percentage points of mandate.

A subset of cases can be given, e.g. `run_refueleu_budget.py B5 pess`.

In [ ]:
!{PYTHON} runs/run_refueleu_budget.py main B5 B75 B15 pess

## 5. Dataset

Exports the 78 runs the paper uses from the raw outputs into `dataset/` (see its README),
which is all `02_results.ipynb` reads. A run added after the paper is declared in `ADDED`
in `helpers/dataset.py` before exporting.

In [ ]:
!{PYTHON} -m helpers.dataset